# 🧪 Notebook 2 — Test des règles individuellement
**Projet : Contrôle qualité des données Proxima**

Ce notebook permet de tester chaque règle de manière isolée,
avant l'assemblage dans le pipeline final.

---

## 0. Imports & chargement

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

CSV_PATH = "data.csv"
df_raw = pd.read_csv(CSV_PATH, sep=None, engine='python')

# Filtre contrats actifs
df = df_raw[
    (df_raw['STOCK'] == 1.0) &
    (df_raw['DT_EFF_RESIL_CNT'].isna())
].copy()

# Correction CP_SITE
df['CP_SITE_FIXED'] = df['CP_SITE'].apply(
    lambda x: str(int(x)).zfill(5) if pd.notna(x) else None
)

print(f"✅ {len(df)} contrats actifs chargés")
print(df[['ID_SITE','PTGST','NOM_CLI','CP_SITE','CP_SITE_FIXED','VILLE_SITE','PAYS_SITE']].head(8))

## Helpers — Fonctions utilitaires partagées

In [ ]:
def make_anomaly(row, code, libelle, gravite, colonne, valeur):
    """Crée un dictionnaire représentant une anomalie détectée."""
    return {
        'id_site'            : row.get('ID_SITE'),
        'ptgst'              : row.get('PTGST'),
        'nu_cnt'             : row.get('NU_CNT'),
        'nom_cli'            : row.get('NOM_CLI'),
        'nom_site'           : row.get('NOM_SITE'),
        'marche'             : row.get('marche'),
        'code_anomalie'      : code,
        'libelle_anomalie'   : libelle,
        'gravite'            : gravite,
        'colonne_concernee'  : colonne,
        'valeur_problematique': str(valeur),
    }

def show_results(anomalies, rule_name):
    """Affiche proprement les résultats d'une règle."""
    if not anomalies:
        print(f"✅ {rule_name} — Aucune anomalie détectée")
        return pd.DataFrame()
    df_out = pd.DataFrame(anomalies)
    print(f"⚠️  {rule_name} — {len(df_out)} anomalie(s) détectée(s)")
    return df_out

print("✅ Helpers définis")

---
## Règle 1 — Présence des champs adresse

**Logique :**
- Anomalie grave  : CP_SITE ou VILLE_SITE manquant
- Anomalie grave  : tous les champs adresse absents
- Anomalie légère : rue absente (mais CP + ville présents)
- Anomalie grave  : PAYS_SITE absent

In [ ]:
def check_rule1(df):
    anomalies = []
    for _, row in df.iterrows():
        cp    = row.get('CP_SITE_FIXED')
        ville = row.get('VILLE_SITE')
        rue   = row.get('RUE_SITE') or row.get('NUM_RUE_SITE')
        pays  = row.get('PAYS_SITE')

        # R1-04 : Pays manquant
        if pd.isna(pays) or str(pays).strip() == '':
            anomalies.append(make_anomaly(row, 'R1-04', 'Pays manquant', 'Grave', 'PAYS_SITE', pays))

        # R1-01 : Adresse entièrement manquante
        if pd.isna(cp) and pd.isna(ville) and pd.isna(rue):
            anomalies.append(make_anomaly(row, 'R1-01', 'Adresse entièrement manquante', 'Grave', 'ADRESSE', 'null'))
            continue  # inutile de vérifier les sous-règles

        # R1-03 : CP ou Ville manquant
        if pd.isna(cp):
            anomalies.append(make_anomaly(row, 'R1-03', 'Code postal manquant', 'Grave', 'CP_SITE', cp))
        if pd.isna(ville) or str(ville).strip() == '':
            anomalies.append(make_anomaly(row, 'R1-03', 'Ville manquante', 'Grave', 'VILLE_SITE', ville))

        # R1-02 : Rue absente (mais CP + ville présents)
        if pd.notna(cp) and pd.notna(ville) and pd.isna(rue):
            anomalies.append(make_anomaly(row, 'R1-02', 'Adresse incomplète (rue absente)', 'Légère', 'RUE_SITE', 'null'))

    return anomalies

r1_anomalies = check_rule1(df)
r1_df = show_results(r1_anomalies, "Règle 1 — Présence adresse")
r1_df

In [ ]:
# Détail par code d'anomalie
if not r1_df.empty:
    print(r1_df['code_anomalie'].value_counts())

---
## Règle 2 — Format et validité du code postal

**Logique :**
- R2-01 : format invalide (pas 5 chiffres)
- R2-02 : département inexistant

In [ ]:
# Référentiel des départements valides (France)
DEPTS_VALIDES = set(
    [str(i).zfill(2) for i in range(1, 96) if i != 20]
    + ['2A', '2B', '971', '972', '973', '974', '976']
)

def check_rule2(df):
    anomalies = []
    for _, row in df.iterrows():
        pays = str(row.get('PAYS_SITE') or '').strip().upper()
        cp   = row.get('CP_SITE_FIXED')

        if pd.isna(cp):
            continue  # déjà couvert par règle 1

        if pays == 'FRANCE' or pays == '':
            # R2-01 : format 5 chiffres
            if not re.fullmatch(r'\d{5}', cp):
                anomalies.append(make_anomaly(row, 'R2-01', 'Format code postal invalide', 'Grave', 'CP_SITE', cp))
                continue

            # R2-02 : département existant
            dept = cp[:2] if cp[:2] != '97' else cp[:3]
            if dept not in DEPTS_VALIDES:
                anomalies.append(make_anomaly(row, 'R2-02', f'Département inexistant ({dept})', 'Grave', 'CP_SITE', cp))

        else:
            # Autres pays : non validé dans cette version
            pass

    return anomalies

r2_anomalies = check_rule2(df)
r2_df = show_results(r2_anomalies, "Règle 2 — Format code postal")
r2_df

In [ ]:
# Test manuel : cas limites
test_cp = ['75001', '04100', '97100', '00000', '99999', '2A000', 'ABCDE', None]
print("Test manuel sur des CPs :")
for cp in test_cp:
    if cp is None:
        print(f"  {str(cp):<8} → ignoré (null)")
        continue
    if not re.fullmatch(r'\d{5}', cp):
        print(f"  {cp:<8} → ❌ Format invalide")
    else:
        dept = cp[:2] if cp[:2] != '97' else cp[:3]
        status = '✅ Valide' if dept in DEPTS_VALIDES else f'❌ Département inexistant ({dept})'
        print(f"  {cp:<8} → {status}")

---
## Règle 3 — Cohérence code postal / ville

**Source de référence :** Base officielle data.gouv.fr (codes postaux → communes)

**Scénarios :**
- A : CP + Ville présents → vérification complète
- B : CP présent, Ville absente → validation CP seul
- C/D : CP absent → règle non applicable

In [ ]:
# Chargement de la base des codes postaux français
# Télécharger depuis : https://www.data.gouv.fr/fr/datasets/base-officielle-des-codes-postaux/
# Format attendu : colonnes 'code_postal' et 'nom_de_la_commune' (ou similaire)

import os

CP_DB_PATH = "laposte_hexasmal.csv"  # à adapter selon le fichier téléchargé

if os.path.exists(CP_DB_PATH):
    cp_db = pd.read_csv(CP_DB_PATH, sep=';', dtype=str)
    print(f"✅ Base codes postaux chargée : {len(cp_db)} entrées")
    print(cp_db.head(3))
else:
    print("⚠️  Fichier de référence non trouvé.")
    print(f"   → Télécharger depuis data.gouv.fr et placer en : {CP_DB_PATH}")
    print("   → La règle 3 ne peut pas tourner sans ce fichier.")
    cp_db = None

In [ ]:
def normalize_str(s):
    """Normalisation pour comparaison : minuscules + espaces superflus."""
    if pd.isna(s):
        return None
    return ' '.join(str(s).lower().strip().split())

def build_cp_index(cp_db, cp_col, ville_col):
    """Construit un dict {cp -> set(villes normalisées)}."""
    index = {}
    for _, row in cp_db.iterrows():
        cp    = str(row[cp_col]).zfill(5)
        ville = normalize_str(row[ville_col])
        if cp not in index:
            index[cp] = set()
        if ville:
            index[cp].add(ville)
    return index

# Adapter les noms de colonnes selon le fichier téléchargé
# Exemple typique data.gouv : 'Code_postal', 'Nom_commune'
# cp_index = build_cp_index(cp_db, cp_col='Code_postal', ville_col='Nom_commune')

print("ℹ️  Adapter build_cp_index() selon les noms de colonnes du fichier téléchargé.")

In [ ]:
def check_rule3(df, cp_index):
    """
    Scénario A : CP + Ville → vérification complète
    Scénario B : CP seul → validation existence CP
    Scénarios C/D : CP absent → non applicable
    """
    anomalies = []
    if cp_index is None:
        print("⚠️  cp_index non disponible — règle 3 ignorée")
        return anomalies

    for _, row in df.iterrows():
        cp    = row.get('CP_SITE_FIXED')
        ville = normalize_str(row.get('VILLE_SITE'))

        if pd.isna(cp):
            continue  # scénario C/D — couvert par règle 1

        cp_exists = cp in cp_index

        if not cp_exists:
            anomalies.append(make_anomaly(row, 'R2-02', 'Code postal inexistant dans la base officielle', 'Grave', 'CP_SITE', cp))
            continue

        if ville:
            # Scénario A — vérification cohérence
            villes_ref = cp_index[cp]
            if ville not in villes_ref:
                detail = f"CP={cp} → villes attendues: {', '.join(list(villes_ref)[:5])}"
                anomalies.append(make_anomaly(row, 'R3-01', f'Incohérence CP/Ville ({detail})', 'Légère', 'VILLE_SITE', ville))
        # Scénario B : CP existe, ville absente → déjà flagué par règle 1, rien à ajouter ici

    return anomalies

# Exemple d'exécution (décommenter quand cp_index est disponible)
# r3_anomalies = check_rule3(df, cp_index)
# r3_df = show_results(r3_anomalies, "Règle 3 — Cohérence CP/Ville")
# r3_df
print("ℹ️  check_rule3() définie. Décommenter l'exécution après chargement de cp_index.")

In [ ]:
# Test manuel : simulation sans la base officielle
print("=== Test manuel cohérence CP / Ville ===")
test_cases = [
    ('75001', 'paris'),
    ('75001', 'lyon'),
    ('69001', 'lyon'),
    ('69001', 'paris'),
    ('99999', 'ville_inexistante'),
    (None,    'bordeaux'),
]
# Simulation avec un mini-référentiel
mini_ref = {'75001': {'paris'}, '69001': {'lyon'}, '13001': {'marseille'}}
for cp, ville in test_cases:
    if cp is None:
        print(f"  CP=None, Ville={ville:<20} → scénario C/D — non applicable")
        continue
    if cp not in mini_ref:
        print(f"  CP={cp}, Ville={ville:<20} → ❌ CP inexistant")
    elif ville in mini_ref[cp]:
        print(f"  CP={cp}, Ville={ville:<20} → ✅ Cohérent")
    else:
        print(f"  CP={cp}, Ville={ville:<20} → ⚠️  Incohérence CP/Ville")

---
## Règle 4 — Validité et cohérence des coordonnées GPS

**Trois scénarios :**
- A : Adresse présente, GPS absent → Valide
- B : Adresse + GPS présents → Vérifier cohérence avec bounding box département
- C : GPS seul, adresse absente → Vérifier que les coords sont dans le pays

In [ ]:
# Bounding boxes France (métropole) par département
# Format : dept_code -> (lat_min, lat_max, lon_min, lon_max)
# Source : calculées à partir des contours GeoJSON officiels (data.gouv.fr)
# Note : valeurs approximatives pour les tests — à remplacer par le fichier généré

BBOX_DEPT = {
    '01': (45.75, 46.52, 4.75, 5.77),  '02': (49.07, 50.07, 2.97, 4.21),
    '13': (43.16, 43.93, 4.23, 5.78),  '31': (42.67, 44.02, 0.60, 3.20),
    '33': (44.20, 45.58, -1.24, 0.30), '34': (43.20, 43.96, 2.95, 4.20),
    '38': (44.70, 45.91, 4.93, 6.40),  '59': (50.03, 51.09, 2.09, 3.91),
    '62': (50.03, 51.00, 1.56, 3.01),  '63': (45.22, 46.06, 2.34, 3.99),
    '69': (45.46, 46.30, 4.24, 5.32),  '75': (48.81, 48.91, 2.22, 2.47),
}

# Bounding box France entière (pour scénario C si PAYS=FRANCE)
BBOX_PAYS = {
    'FRANCE': (41.30, 51.10, -5.10,  9.60),
    'BELGIQUE': (49.50, 51.50,  2.55,  6.40),
    'LUXEMBOURG': (49.44, 50.18,  5.73,  6.53),
}

def get_dept_from_cp(cp):
    if cp is None or len(cp) < 2:
        return None
    return cp[:3] if cp[:2] == '97' else cp[:2]

def coords_in_bbox(lat, lon, bbox):
    lat_min, lat_max, lon_min, lon_max = bbox
    return lat_min <= lat <= lat_max and lon_min <= lon <= lon_max

print("✅ Bounding boxes définies")
print(f"   {len(BBOX_DEPT)} département(s) | {len(BBOX_PAYS)} pays")
print()
print("⚠️  Note : ces bounding boxes sont approximatives pour les tests.")
print("   Le fichier complet sera généré via scripts/build_dept_bboxes.py")

In [ ]:
def check_rule4(df):
    anomalies = []
    for _, row in df.iterrows():
        cp    = row.get('CP_SITE_FIXED')
        ville = row.get('VILLE_SITE')
        pays  = str(row.get('PAYS_SITE') or '').strip().upper()
        x     = row.get('COORD_X_SITE')  # longitude
        y     = row.get('COORD_Y_SITE')  # latitude

        has_address = pd.notna(cp) and pd.notna(ville)
        has_gps     = pd.notna(x) and pd.notna(y)

        # Scénario A : adresse présente, GPS absent → valide, rien à faire
        if has_address and not has_gps:
            continue

        # Scénario B : adresse + GPS présents
        if has_address and has_gps:
            dept = get_dept_from_cp(cp)
            if dept and dept in BBOX_DEPT:
                bbox = BBOX_DEPT[dept]
                if not coords_in_bbox(float(y), float(x), bbox):
                    anomalies.append(make_anomaly(
                        row, 'R4-01',
                        f'GPS hors bounding box du département {dept}',
                        'Légère', 'COORD_X_SITE / COORD_Y_SITE', f'lon={x}, lat={y}'
                    ))
            else:
                # Département non référencé → fallback sur bbox pays
                if pays in BBOX_PAYS:
                    if not coords_in_bbox(float(y), float(x), BBOX_PAYS[pays]):
                        anomalies.append(make_anomaly(
                            row, 'R4-01',
                            f'GPS hors des limites du pays {pays} (département {dept} non référencé)',
                            'Légère', 'COORD_X_SITE / COORD_Y_SITE', f'lon={x}, lat={y}'
                        ))

        # Scénario C : GPS présent, adresse absente
        if has_gps and not has_address:
            if pays in BBOX_PAYS:
                if not coords_in_bbox(float(y), float(x), BBOX_PAYS[pays]):
                    anomalies.append(make_anomaly(
                        row, 'R4-02',
                        f'GPS hors des limites du pays {pays}',
                        'Grave', 'COORD_X_SITE / COORD_Y_SITE', f'lon={x}, lat={y}'
                    ))
            else:
                anomalies.append(make_anomaly(
                    row, 'R4-02',
                    f'GPS présent mais pays inconnu ({pays}) — impossible de valider',
                    'Grave', 'PAYS_SITE', pays
                ))

    return anomalies

r4_anomalies = check_rule4(df)
r4_df = show_results(r4_anomalies, "Règle 4 — GPS")
r4_df

In [ ]:
# Test manuel bounding boxes
print("=== Tests manuels bounding boxes ===")
test_gps = [
    ('75001', 'FRANCE', 48.860, 2.350),    # Paris centre → dans 75
    ('75001', 'FRANCE', 43.300, 5.400),    # Marseille → hors 75 mais dans France
    ('69001', 'FRANCE', 43.300, 5.400),    # Marseille → hors 69
    ('69001', 'FRANCE', 45.750, 4.850),    # Lyon → dans 69
    (None,    'FRANCE', 48.860, 2.350),    # Paris, sans adresse → scénario C
    (None,    'FRANCE', 51.500, -0.120),   # Londres → hors France
]

for cp, pays, lat, lon in test_gps:
    has_addr = cp is not None
    dept = get_dept_from_cp(cp) if cp else None

    if has_addr:
        if dept and dept in BBOX_DEPT:
            valid = coords_in_bbox(lat, lon, BBOX_DEPT[dept])
            status = '✅ Cohérent' if valid else f'⚠️  Hors bbox dept {dept}'
        else:
            status = f'⚠️  Dept {dept} non référencé'
    else:
        # Scénario C
        valid = coords_in_bbox(lat, lon, BBOX_PAYS.get(pays, (0,0,0,0)))
        status = f'✅ Dans {pays}' if valid else f'❌ Hors {pays}'

    print(f"  CP={str(cp):<6} lat={lat:.3f} lon={lon:.3f}  → {status}")

---
## Règle 5 — Présence des données techniques (capitaux ou superficie)

**Logique :** Au moins un des champs doit être renseigné et > 0 :
- `SURFACE` ou `SURFACE_DPDCE`
- OU n'importe quel champ `CAPITAUX_*` (hors `CAPITAUX_TT` à confirmer)

In [ ]:
CAPITAUX_COLS = [
    'CAPITAUX_MURS_BATIMENTS',
    'CAPITAUX_CONTENU',
    'CAPITAUX_MATERIEL',
    'CAPITAUX_MARCHANDISES',
    # 'CAPITAUX_TT',  # exclu : toujours 0 dans l'échantillon — à confirmer
]
SURFACE_COLS = ['SURFACE', 'SURFACE_DPDCE']

def check_rule5(df):
    anomalies = []
    for _, row in df.iterrows():
        has_surface  = any(pd.notna(row.get(c)) and float(row.get(c) or 0) > 0 for c in SURFACE_COLS if c in row.index)
        has_capitaux = any(pd.notna(row.get(c)) and float(row.get(c) or 0) > 0 for c in CAPITAUX_COLS if c in row.index)

        if not has_surface and not has_capitaux:
            anomalies.append(make_anomaly(
                row, 'R5-01',
                'Données techniques manquantes (capitaux et superficie absents)',
                'Grave', 'CAPITAUX / SURFACE', 'null'
            ))
    return anomalies

r5_anomalies = check_rule5(df)
r5_df = show_results(r5_anomalies, "Règle 5 — Données techniques")
r5_df.head(10)

In [ ]:
# Détail : quelles colonnes sont remplies pour les lignes sans anomalie R5
ok_lines = df.copy()
ok_mask = ~ok_lines['ID_SITE'].isin(r5_df['id_site']) if not r5_df.empty else pd.Series([True]*len(ok_lines))
print("=== Colonnes techniques remplies pour les lignes VALIDES ===")
for col in SURFACE_COLS + CAPITAUX_COLS:
    if col in df.columns:
        n = df[ok_mask][col].notna().sum()
        print(f"  {col:<35} : {n} renseignées")

---
## Synthèse — Toutes les règles

In [ ]:
import datetime

all_anomalies = r1_anomalies + r2_anomalies + r4_anomalies + r5_anomalies
# + r3_anomalies  # décommenter quand la base CP est disponible

df_all = pd.DataFrame(all_anomalies)
if not df_all.empty:
    df_all['date_analyse'] = datetime.date.today().isoformat()

print(f"Total anomalies détectées : {len(df_all)}")
print()
if not df_all.empty:
    print("=== Répartition par règle ===")
    print(df_all['code_anomalie'].value_counts())
    print()
    print("=== Répartition par gravité ===")
    print(df_all['gravite'].value_counts())

In [ ]:
# Aperçu final du rapport
if not df_all.empty:
    df_all[['id_site','ptgst','nom_cli','code_anomalie','libelle_anomalie','gravite','colonne_concernee','valeur_problematique']].head(20)

In [ ]:
# Export test (optionnel)
if not df_all.empty:
    out_path = f"test_rapport_anomalies_{datetime.date.today().isoformat()}.xlsx"
    df_all.to_excel(out_path, index=False)
    print(f"✅ Rapport de test exporté : {out_path}")
else:
    print("Aucune anomalie à exporter.")